# Surface-defect inspection (PatchCore) on Colab

Runs the whole project on a free Colab CPU runtime (a GPU is **not** needed: inference uses ONNX Runtime on CPU).
Run the cells top to bottom. Section **C** (MVTec AD) and **D** (your own photos) are the real evaluations;
**B** is a quick synthetic pipeline check (not MVTec data).

## A. Get the project and build the backbone

In [ ]:
# Option 1 (GitHub): the project repository. Set to "" to upload the zip instead (works for private repos).
REPO_URL = "https://github.com/imu-002/surface-defect-patchcore.git"

import os
if REPO_URL:
    !git clone -q {REPO_URL} /content/project
else:
    # Option 2: upload surface_defect_project.zip from your computer when asked
    from google.colab import files
    up = files.upload()
    zipname = next(iter(up))
    !unzip -q -o "{zipname}" -d /content
    !mv /content/surface_defect_project /content/project
%cd /content/project

In [ ]:
!pip -q install onnxruntime scikit-image opencv-python-headless
!python scripts/01_build_backbone.py        # downloads ImageNet weights, converts to ONNX, self-checks (about 1 minute)

## B. Quick pipeline check on synthetic data (about 5 minutes; NOT MVTec results)

In [ ]:
!python scripts/00_make_standin_dataset.py --out data/standin --train 60 --test-good 24 --per-defect 8 > /dev/null
!python scripts/run_experiments.py --data data/standin --categories fabric wood --out results/standin
!python scripts/aggregate.py --runs results/standin --names Synthetic --out results/standin_report > /dev/null

## C. MVTec AD (the real benchmark)

1. Download **mvtec_anomaly_detection.tar.xz** from <https://www.mvtec.com/company/research/datasets/mvtec-ad>
   (accept the licence; non-commercial use) and upload it to your Google Drive.
2. Set the path below, then run. Expect roughly 3-15 minutes per category on Colab's 2 CPU cores.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MVTEC_TAR = '/content/drive/MyDrive/mvtec_anomaly_detection.tar.xz'   # <- change if needed
!mkdir -p /content/mvtec && tar -xf "{MVTEC_TAR}" -C /content/mvtec
!ls /content/mvtec

In [ ]:
# Categories named in the project: carpet leather wood tile grid. Add more if you like.
CATS = "carpet leather wood tile grid"
!python scripts/run_experiments.py --data /content/mvtec --categories {CATS} --out results/mvtec
!python scripts/aggregate.py --runs results/mvtec --names MVTec --out results/mvtec_report

## D. Your own fabric / leather / wood photos

Arrange them like this and zip the folder `my_materials`:

```
my_materials/fabric/train/good/*.jpg            30-200 defect-free photos
my_materials/fabric/test/good/*.jpg             extra good photos
my_materials/fabric/test/<defect_type>/*.jpg    one folder per defect type
my_materials/fabric/ground_truth/<defect_type>/<name>_mask.png     (optional)
```
Same layout for leather and wood. Use the same camera, lighting and distance for every photo.

In [ ]:
from google.colab import files
up = files.upload()                      # choose my_materials.zip
!unzip -q -o my_materials.zip -d /content
!python scripts/run_experiments.py --data /content/my_materials --out results/local
!python scripts/aggregate.py --runs results/local --names Local --out results/local_report

## E. Look at the results and download them

In [ ]:
import glob
from IPython.display import Image, display, Markdown
run = 'results/mvtec' if os.path.exists('results/mvtec') else 'results/standin'
for f in sorted(glob.glob(run + '/qualitative_*.png')):
    display(Image(f))
rep = run + '_report' if os.path.exists(run + '_report') else 'results/standin_report'
display(Markdown(open(rep + '/summary.md').read()))

In [ ]:
!zip -qr /content/my_results.zip results -x "*/models/*"
from google.colab import files
files.download('/content/my_results.zip')